In [1]:
%run functions.ipynb

In [2]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = load_data()
y_train = df.yt_true.iloc[:-24].values.astype(float)

candidatos = [((0, 1, 1), (0, 1, 1, 12)), ((1, 1, 1), (0, 1, 1, 12)), ((2, 1, 0), (0, 1, 1, 12))]

for order, seasonal_order in candidatos:
    result = SARIMAX(y_train, order=order, seasonal_order=seasonal_order).fit(disp=False)
    print(order, seasonal_order, "AIC =", round(result.aic, 1))

(0, 1, 1) (0, 1, 1, 12) AIC = 2973.3
(1, 1, 1) (0, 1, 1, 12) AIC = 2973.3
(2, 1, 0) (0, 1, 1, 12) AIC = 2985.2


In [3]:
# Los dos primeros empatan en AIC; se elige el mas simple (menos parametros)

result = SARIMAX(y_train, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)

yt_pred = np.r_[result.fittedvalues, result.forecast(24)]

# Las primeras 13 predicciones no son validas: el modelo aun no tiene
# historia suficiente para las diferencias 1 y 12.
yt_pred[:13] = np.nan

df["yt_pred_sarima"] = yt_pred

plot_time_series(df[["yt_true", "yt_pred_sarima"]])

<Figure size 1200x400 with 1 Axes>

In [4]:
result.summary().tables[1]

,coef,std err,z,P>|z|,[0.025,0.975]
ma.L1,-0.5773,0.051,-11.386,0.000,-0.677,-0.478
ma.S.L12,-0.7734,0.046,-16.661,0.000,-0.864,-0.682
sigma2,5.534e+04,4082.878,13.554,0.000,4.73e+04,6.33e+04


In [6]:
metrics = compute_evaluation_metrics(df)
save_forecasts(df)
save_metrics(metrics)
metrics

,Metrics,yt_pred_sarima
0,MSE Train,57331.27
1,MSE Test,140040.00
2,MAE Train,174.32
3,MAE Test,304.02
